# Post-1990 Democratizers Panel Dataset (1990-2022): Demo Notebook

## Description

This notebook demonstrates the **Post-1990 Democratizers Panel Dataset** creation and analysis pipeline. The dataset contains **1,552 country-year observations** from **48 post-1990 democratizers** covering **1990-2022**.

### Variables
1. **Gini coefficient** (0-1 scale, mean=0.39) — from World Bank PIP via OWID
2. **V-Dem Electoral Democracy Index** (0-1 scale, mean=0.61) — V-Dem v13 via OWID
3. **Mean years of schooling** (0-17 years, mean=8.3) — Barro-Lee via OWID
4. **Social protection spending** (% GDP, mean=13.2%) — OECD via OWID

### Data Processing
- **Multiple Imputation by Chained Equations (MICE)** via sklearn IterativeImputer with bounds constraints
- **Region dummy variables** for regression analysis
- **5-fold cross-validation** splits included

### Task Format
The data is formatted for an **experiment pipeline** as a **regression task**:
- **Input**: Gini + education + social spending + region dummies (JSON string)
- **Output**: V-Dem democracy index (string)

### Sources
- Our World in Data (OWID): World Bank PIP, V-Dem, Barro-Lee education, OECD social spending


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
# loguru is not in the pre-installed list
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.experimental import enable_iterative_imputer  # Enable MICE imputer
from sklearn.impute import IterativeImputer

# NumPy 2.0 compatibility shims (if needed)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

print("All imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-90d6bf-the-triple-shield-revisited-education-we/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL load failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: 
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os
data = load_data()
print("Data loaded successfully!")
print(f"Dataset name: {data['datasets'][0]['dataset']}")
print(f"Number of examples: {len(data['datasets'][0]['examples'])}")
print(f"Metadata: {data['metadata']}")


In [ ]:
# =============================================================================
# CONFIGURATION - Set tunable parameters here
# =============================================================================

# MICE imputation parameters (from merge_final.py)
MICE_MAX_ITER = 5  # MINIMUM: Original is 50, use 5 for fast demo
MICE_RANDOM_STATE = 42

# Data processing parameters
N_EXAMPLES = 15  # Number of examples to process (mini dataset size)
N_COUNTRIES = 5   # Number of countries to include

# Bounds for MICE imputation (same as original)
BOUNDS = {
    'gini': (0, 1),
    'v2x_polyarchy': (0, 1),
    'education': (0, 20),
    'social_spending': (0, 50)
}

print("Configuration set:")
print(f"  MICE max iterations: {MICE_MAX_ITER}")
print(f"  Number of examples: {N_EXAMPLES}")
print(f"  Bounds: {BOUNDS}")


## Understanding the Data Structure

The demo data is already in the **experiment pipeline format** (exp_sel_data_out.json schema):

- `input`: JSON string of feature values (gini, education, social_spending, region dummies)
- `output`: V-Dem democracy index as string
- `metadata_*`: Country, year, fold for cross-validation, feature names, task type

Let's parse and explore the data structure.


In [ ]:
# Parse the demo data
examples = data['datasets'][0]['examples']

# Convert to DataFrame for easier analysis
parsed_data = []
for ex in examples:
    input_dict = json.loads(ex['input'])
    row = {
        'country': ex['metadata_country'],
        'country_code': ex['metadata_country_code'],
        'year': ex['metadata_year'],
        'fold': ex['metadata_fold'],
        'v2x_polyarchy': float(ex['output']),
        **input_dict  # Unpack all input features
    }
    parsed_data.append(row)

df = pd.DataFrame(parsed_data)
print("Data shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)


## Data Exploration

Let's explore the key variables in the dataset:
- Distribution of democracy scores (v2x_polyarchy)
- Gini coefficient (inequality)
- Education levels
- Social spending


In [ ]:
# Summary statistics
print("=== Summary Statistics ===")
print("\nDemocracy Index (v2x_polyarchy):")
print(df['v2x_polyarchy'].describe())

print("\nGini Coefficient:")
gini_cols = [c for c in df.columns if 'gini' in c.lower() and c in df.columns]
if gini_cols:
    print(df[gini_cols].describe())

# Check regions
region_cols = [c for c in df.columns if c.startswith('region_')]
print("\nRegion dummies:")
print(df[region_cols].sum())


## Reproducing the Data Processing Pipeline

Now let's reproduce key steps from the original `merge_final.py` script:

1. **MICE Imputation** - Show how missing values are imputed
2. **Region dummy creation** - Convert region to dummies
3. **Data validation** - Check bounds and data quality

Since our mini demo data is already processed, we'll demonstrate the MICE imputation on a small synthetic dataset.


In [ ]:
# Demonstrate MICE imputation with a small example
# Create a small dataset with some missing values to demonstrate MICE

np.random.seed(MICE_RANDOM_STATE)

# Create synthetic data with missing values
demo_df = pd.DataFrame({
    'gini': np.random.uniform(0.25, 0.60, 10),
    'v2x_polyarchy': np.random.uniform(0.2, 0.9, 10),
    'education': np.random.uniform(4, 14, 10),
    'social_spending': np.random.uniform(3, 22, 10)
})

# Introduce some missing values
demo_df.loc[2, 'gini'] = np.nan
demo_df.loc[5, 'education'] = np.nan
demo_df.loc[7, 'social_spending'] = np.nan

print("Dataset with missing values:")
print(demo_df)
print("\nMissing values per column:")
print(demo_df.isnull().sum())

# Run MICE imputation
print("\n=== Running MICE Imputation ===")
impute_vars = ['gini', 'v2x_polyarchy', 'education', 'social_spending']
X = demo_df[impute_vars].copy()

imputer = IterativeImputer(max_iter=MICE_MAX_ITER, random_state=MICE_RANDOM_STATE, sample_posterior=True)
X_imp = imputer.fit_transform(X)

# Apply bounds
for i, var in enumerate(impute_vars):
    X_imp[:, i] = np.clip(X_imp[:, i], BOUNDS[var][0], BOUNDS[var][1])

demo_df_imputed = demo_df.copy()
for i, var in enumerate(impute_vars):
    demo_df_imputed[var] = X_imp[:, i]

print("\nDataset after MICE imputation:")
print(demo_df_imputed)
print("\nImputed values:")
print(f"  Gini (row 2): {demo_df_imputed.loc[2, 'gini']:.4f} (was NaN)")
print(f"  Education (row 5): {demo_df_imputed.loc[5, 'education']:.4f} (was NaN)")
print(f"  Social spending (row 7): {demo_df_imputed.loc[7, 'social_spending']:.4f} (was NaN)")


## Data Visualization

Let's create visualizations to understand the relationships in the data:

1. **Democracy vs Inequality** (Gini coefficient)
2. **Democracy vs Education**
3. **Democracy vs Social Spending**
4. **Regional patterns**


In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Post-1990 Democratizers: Key Relationships', fontsize=14, fontweight='bold')

# Plot 1: Democracy vs Gini
ax1 = axes[0, 0]
gini_col = [c for c in df.columns if 'gini' in c.lower()][0]
ax1.scatter(df[gini_col], df['v2x_polyarchy'], alpha=0.6, s=80, color='darkred')
ax1.set_xlabel('Gini Coefficient (Inequality)', fontsize=11)
ax1.set_ylabel('V-Dem Democracy Index', fontsize=11)
ax1.set_title('Democracy vs Inequality', fontsize=12)
ax1.grid(True, alpha=0.3)

# Plot 2: Democracy vs Education
ax2 = axes[0, 1]
edu_col = [c for c in df.columns if 'education' in c.lower()][0]
ax2.scatter(df[edu_col], df['v2x_polyarchy'], alpha=0.6, s=80, color='darkblue')
ax2.set_xlabel('Mean Years of Schooling', fontsize=11)
ax2.set_ylabel('V-Dem Democracy Index', fontsize=11)
ax2.set_title('Democracy vs Education', fontsize=12)
ax2.grid(True, alpha=0.3)

# Plot 3: Democracy vs Social Spending
ax3 = axes[1, 0]
social_col = [c for c in df.columns if 'social' in c.lower()][0]
ax3.scatter(df[social_col], df['v2x_polyarchy'], alpha=0.6, s=80, color='darkgreen')
ax3.set_xlabel('Social Protection Spending (% GDP)', fontsize=11)
ax3.set_ylabel('V-Dem Democracy Index', fontsize=11)
ax3.set_title('Democracy vs Social Spending', fontsize=12)
ax3.grid(True, alpha=0.3)

# Plot 4: Regional distribution of democracy
ax4 = axes[1, 1]
region_cols = [c for c in df.columns if c.startswith('region_')]
regions = []
democracy_scores = []
for idx, row in df.iterrows():
    for r in region_cols:
        if row[r] == 1.0:
            regions.append(r.replace('region_', ''))
            democracy_scores.append(row['v2x_polyarchy'])
            break

region_df = pd.DataFrame({'region': regions, 'democracy': democracy_scores})
region_means = region_df.groupby('region')['democracy'].mean().sort_values(ascending=True)
ax4.barh(region_means.index, region_means.values, color='purple', alpha=0.7)
ax4.set_xlabel('Mean V-Dem Democracy Index', fontsize=11)
ax4.set_title('Mean Democracy by Region', fontsize=12)
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Print correlation matrix
print("\n=== Correlation Matrix ===")
numeric_cols = [gini_col, edu_col, social_col, 'v2x_polyarchy']
corr_df = df[numeric_cols].copy()
print(corr_df.corr())


## Simple Regression Analysis Demo

Let's run a simple linear regression to demonstrate the **prediction task**:
- **Predict**: V-Dem democracy index
- **Using**: Gini + education + social spending + region dummies

This reproduces the experiment pipeline format.


In [ ]:
# Simple regression demo
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Prepare features and target
feature_cols = [c for c in df.columns if c not in ['country', 'country_code', 'year', 'fold', 'v2x_polyarchy']]
X = df[feature_cols].values
y = df['v2x_polyarchy'].values

print("Feature columns used:", feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Predictions
y_pred = model.predict(X)

# Evaluate
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("\n=== Regression Results (on demo data) ===")
print(f"MSE: {mse:.6f}")
print(f"R²: {r2:.6f}")
print(f"\nCoefficients:")
for name, coef in zip(feature_cols, model.coef_):
    print(f"  {name}: {coef:.6f}")
print(f"\nIntercept: {model.intercept_:.6f}")

# Show predictions vs actual
results_df = pd.DataFrame({
    'country': df['country'],
    'year': df['year'],
    'actual': y,
    'predicted': y_pred,
    'error': y - y_pred
})
print("\n=== Predictions vs Actual ===")
print(results_df[['country', 'year', 'actual', 'predicted', 'error']].to_string())


## Summary

This notebook demonstrated the **Post-1990 Democratizers Panel Dataset** pipeline:

### Key Steps
1. ✅ **Data Loading**: Loaded mini demo data from GitHub URL (with local fallback)
2. ✅ **Data Exploration**: Examined structure, summary statistics, regional distribution
3. ✅ **MICE Imputation Demo**: Demonstrated multiple imputation on synthetic data
4. ✅ **Visualization**: Created scatter plots and regional comparisons
5. ✅ **Regression Demo**: Simple linear regression on the demo data

### Dataset Characteristics
- **1,552 country-year observations** (48 countries, 1990-2022)
- **Variables**: Gini, V-Dem index, education, social spending, region dummies
- **Task**: Regression (predict democracy from inequality + controls)
- **Processing**: MICE imputation with bounds constraints

### Next Steps for Full Analysis
- Scale up to full dataset (1,552 examples)
- Run proper train/test splits using the provided folds
- Compare different regression models (OLS, Ridge, Lasso, etc.)
- Add fixed effects for countries and years
- Conduct robustness checks

### Files Generated
- `mini_demo_data.json`: Curated subset for demo (15 examples)
- This notebook: Reproducible demo of the data pipeline
